# ARIMAX

## Подготовка

In [1]:
# Работа с данными
import pandas as pd
import numpy as np

# Визуализация
import matplotlib.pyplot as plt
import plotly.express as px

# Тесты на стационарность
from statsmodels.tsa.stattools import adfuller, kpss

# Тест на автокорреляцию
from statsmodels.stats.diagnostic import acorr_ljungbox

# Анализ автокорреляции
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Модели ARIMA
from statsmodels.tsa.arima.model import ARIMA

# Декомпозиция
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.seasonal import STL, MSTL

import warnings
warnings.filterwarnings("ignore")

In [2]:
!pip install pmdarima

# авто ARIMA
import pmdarima as pm
from pmdarima import auto_arima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 10.6 MB/s eta 0:00:00


In [3]:
def adf_test(timeseries):
    print("Results of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=[
            "Test Statistic",
            "p-value",
            "#Lags Used",
            "Number of Observations Used",
        ],
    )
    for key, value in dftest[4].items():
        dfoutput["Critical Value (%s)" % key] = value
    print(dfoutput)

In [4]:
def kpss_test(timeseries):
    print("Results of KPSS Test:")
    kpsstest = kpss(timeseries, regression="c", nlags="auto")
    kpss_output = pd.Series(
        kpsstest[0:3], index=["Test Statistic", "p-value", "Lags Used"]
    )
    for key, value in kpsstest[3].items():
        kpss_output["Critical Value (%s)" % key] = value
    print(kpss_output)

## Данные по акциям

Мы снова будем анализировать цены закрытия акций Microsoft. Данные можно скачать [здесь](https://www.nasdaq.com/market-activity/stocks/msft/historical?page=1&rows_per_page=10&timeline=y5).

Рассмотрим акции Microsoft за 5 лет: у нас 1255 наблюдений с 18.11.2020 по 17.11.2025.

In [8]:
stocks = pd.read_csv('microsoft_new.csv', parse_dates=['Date'])

In [9]:
stocks.head()

,Date,Close/Last,Volume,Open,High,Low
0,2025-11-17,$507.49,19092750,$508.45,$512.12,$504.91
1,2025-11-14,$510.18,28505750,$498.23,$511.60,$497.44
2,2025-11-13,$503.29,25273110,$510.31,$513.50,$501.29
3,2025-11-12,$511.14,26574850,$509.355,$511.67,$499.1201
4,2025-11-11,$508.68,17980020,$504.80,$509.60,$502.3488


Немного предобработки.

In [10]:
stocks.rename(columns={'Date':'ds', 'Close/Last':'y'}, inplace=True)
stocks['y'] = stocks['y'].apply(lambda x: x[1:]).astype(float)

stocks = stocks[['ds','y']].sort_values(by='ds').set_index('ds')

In [11]:
stocks.head()

,y
ds,
2020-11-18,211.08
2020-11-19,212.42
2020-11-20,210.39
2020-11-23,210.11
2020-11-24,213.86


Вспомним, как выглядят данные.

In [12]:
fig = px.line(stocks, x=stocks.index, y=stocks['y'],
              title="Microsoft closing prices (daily)")

fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Сегодня будем строить прогноз out-of-sample, поэтому разделим данные на train и test.

In [13]:
train_size = int(len(stocks) * 0.9)
train, test = stocks[:train_size], stocks[train_size:]

In [14]:
fig = px.line(title="Microsoft closing prices (daily)")
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test', line=dict(color='green'))

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

На занятиях 5–6 мы начали говорить о регрессорах (предикторах), которые позволяют корректировать прогноз с учётом внешних факторов. Давайте освежим теорию, а затем перейдём к моделям, работающим с регрессорами.

## Предикторы (регрессоры)

### Dummy регрессоры

🐎 **Что такое dummy регрессоры?**

Для наших данных по акциям возьмём два dummy регрессора, которые могли существенно повлиять на цену.

**Рост процентных ставок (2022)**

* Начало: декабрь 2021 года, когда ФРС США резко подняла ставки — самый быстрый рост с 1980-х (примерно с 0% до 4–4,5%, [источник](https://tradingeconomics.com/united-states/interest-rate)).

* Конец: январь 2023 года.

**Бум ИИ (2025 — по настоящее время)**

* Начало: около января 2023 года, когда Microsoft объявила о многолетних инвестициях в OpenAI на $10 млрд ([источник](https://blogs.microsoft.com/blog/2023/01/23/microsoftandopenaiextendpartnership/)).
* Конец: эффект продолжается и в последующие годы.

Далее создадим dummy регрессор для периода роста процентных ставок.

In [15]:
date_range = pd.date_range(start=stocks.index.min(), end=stocks.index.max())

# Создаём DataFrame
regressors = pd.DataFrame(date_range, columns=["ds"])

regressors

,ds
0,2020-11-18
1,2020-11-19
2,2020-11-20
3,2020-11-21
4,2020-11-22
...,...
1821,2025-11-13
1822,2025-11-14
1823,2025-11-15
1824,2025-11-16


In [16]:
# Период роста ставок
rising_rates_start = "2020-12-01"
rising_rates_end = "2023-01-31"

# Помечаем даты роста ставок единицей, остальные — нулём
regressors['rising_interest_rates'] = ((regressors['ds'] >= rising_rates_start) \
                            & (regressors['ds'] <= rising_rates_end)).astype(int)

regressors = regressors[~regressors['ds'].dt.weekday.isin([5, 6])] # убираем выходные
regressors.set_index('ds', inplace=True)

regressors = regressors.join(stocks).dropna()['rising_interest_rates'] # в stocks есть несколько пропусков
regressors = pd.DataFrame(regressors)

regressors

,rising_interest_rates
ds,
2020-11-18,0
2020-11-19,0
2020-11-20,0
2020-11-23,0
2020-11-24,0
...,...
2025-11-11,0
2025-11-12,0
2025-11-13,0


Помните, для прогноза вне выборки нужно заранее подготовить регрессоры за тот же период. Посмотрим, как это выглядит.

In [17]:
fig = px.line(title="Rising interest rates dummy regressor")
fig.add_scatter(x=regressors[:train_size].index, y=regressors[:train_size]['rising_interest_rates'], mode='lines', name='Rising rates for train', line=dict(color='blue'))
fig.add_scatter(x=regressors[train_size:].index, y=regressors[train_size:]['rising_interest_rates'], mode='lines', name='Rising rates for forecast', line=dict(color='green'))

fig.update_layout(template='plotly_white', width=1000, height=400)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

Теперь создадим аналогичный регрессор для бума ИИ.

Бум ИИ можно условно разделить на несколько волн:
- ранняя фаза (2017–2020), когда появились прорывы вроде трансформеров;
- профессиональный хайп (2020–2022) с появлением GPT-3 и диффузионных моделей;
- выход в массы после релиза ChatGPT в ноябре 2022;
- всплеск инвестиций и развития инфраструктуры в 2023–2024 годах (облачные провайдеры, производители GPU);
- с 2024–2025 годов — этап коммерциализации, когда ИИ встраивается в корпоративное ПО.

С 2025-го формируется новая волна, связанная с автономными агентами и робототехникой.

Для нашего урока рассмотрим 2 фазы:

1. 2023-01-01 — 2025-01-01 — фаза «разогрева».
2. 2025-01-01 — 2027-01-01 — предполагаем, что текущая волна будет иметь влияние ещё 2 года.

In [18]:
# Периоды бума ИИ
ai_boom_start = "2023-01-01"
ai_boom_end = "2025-01-01"
new_ai_boom_end = "2027-01-01"

# Помечаем даты бума ИИ и новой волны единицей, остальные — нулём
regressors['ai_boom'] = ((regressors.index >= ai_boom_start) \
                            & (regressors.index <= ai_boom_end)).astype(int)
regressors['new_ai_boom'] = ((regressors.index >= ai_boom_end) \
                            & (regressors.index <= new_ai_boom_end)).astype(int)

regressors = regressors[~regressors.index.weekday.isin([5, 6])] # убираем выходные

regressors = regressors.join(stocks).dropna()[['rising_interest_rates', 'ai_boom', 'new_ai_boom']] # в stocks есть несколько пропусков
regressors = pd.DataFrame(regressors)

regressors

,rising_interest_rates,ai_boom,new_ai_boom
ds,,,
2020-11-18,0,0,0
2020-11-19,0,0,0
2020-11-20,0,0,0
2020-11-23,0,0,0
2020-11-24,0,0,0
...,...,...,...
2025-11-11,0,0,1
2025-11-12,0,0,1
2025-11-13,0,0,1


Построим график и посмотрим, как это выглядит.

In [19]:
fig = px.line(title="AI boom dummy regressor")
fig.add_scatter(x=regressors[:train_size].index, y=regressors[:train_size]['ai_boom'], mode='lines', name='AI boom for train', line=dict(color='blue'))
fig.add_scatter(x=regressors[train_size:].index, y=regressors[train_size:]['ai_boom'], mode='lines', name='AI boom for forecast', line=dict(color='green'))

fig.update_layout(template='plotly_white', width=1000, height=400)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

In [20]:
fig = px.line(title="New wave of AI boom dummy regressor")
fig.add_scatter(x=regressors[:train_size].index, y=regressors[:train_size]['new_ai_boom'], mode='lines', name='New wave of AI boom for train', line=dict(color='blue'))
fig.add_scatter(x=regressors[train_size:].index, y=regressors[train_size:]['new_ai_boom'], mode='lines', name='New wave of AI boom for forecast', line=dict(color='green'))

fig.update_layout(template='plotly_white', width=1000, height=400)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

### Непрерывные регрессоры

**Непрерывные регрессоры** — это независимые переменные в регрессионной модели, принимающие любые числовые значения в диапазоне. В отличие от dummy переменных, они описывают реальные измерения или величины. Примеры: экономические индикаторы, температура, рекламные бюджеты и другие внешние факторы, меняющиеся со временем.

Загрузим полный датасет с информацией об акциях Microsoft.

In [21]:
stocks_new = pd.read_csv('microsoft_new.csv', parse_dates=['Date'])
stocks_new.head()

,Date,Close/Last,Volume,Open,High,Low
0,2025-11-17,$507.49,19092750,$508.45,$512.12,$504.91
1,2025-11-14,$510.18,28505750,$498.23,$511.60,$497.44
2,2025-11-13,$503.29,25273110,$510.31,$513.50,$501.29
3,2025-11-12,$511.14,26574850,$509.355,$511.67,$499.1201
4,2025-11-11,$508.68,17980020,$504.80,$509.60,$502.3488


Возьмём цену открытия как непрерывный числовой предиктор. Понятно, что в реальных задачах редко удаётся подобрать регрессор, который идеально описывает целевую переменную.

In [22]:
stocks_new.rename(columns={'Date':'ds', 'Close/Last':'y', 'Open':'x'}, inplace=True)
stocks_new['y'] = stocks_new['y'].apply(lambda x: x[1:]).astype(float)
stocks_new['x'] = stocks_new['x'].apply(lambda x: x[1:]).astype(float)

stocks_new = stocks_new[['ds','y', 'x']].sort_values(by='ds').set_index('ds')

In [23]:
fig = px.line(title="Microsoft closing and open prices (daily)")
fig.add_scatter(x=stocks_new.index, y=stocks_new['y'], mode='lines', name='closing price', line=dict(color='blue'))
fig.add_scatter(x=stocks_new.index, y=stocks_new['x'], mode='lines', name='open price', line=dict(color='red'))

fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

## Модель ARIMAX

**Формула**

$y_{t} = \alpha_{0} + \alpha_{1}y'_{t-1} + ... + \alpha_{p}y'_{t-p} + \beta_{1}\epsilon_{t-1} + ... + \beta_{q}\epsilon_{t-q} + \gamma_1*x_{1,t} + ... + \gamma_r*x_{r,t} + \epsilon_{t}$, где $\epsilon_{t}$ — белый шум, $y'_t$ — дифференцированный ряд, $x_{r,t}$ — предиктор.

Модель ARIMAX расширяет ARIMA за счёт включения внешних регрессоров (экзогенных переменных), которые влияют на зависимый временной ряд.
 Буква **X** в ARIMAX обозначает экзогенные переменные — дополнительные факторы, влияющие на целевой ряд, но не зависящие от него напрямую. Это могут быть экономические индикаторы, погодные данные, социальные события и любые другие внешние воздействия.

Спрогнозируем ряд с помощью ARIMAX и сравним модель с регрессорами и без них.

In [24]:
forecast_size = len(stocks) - train_size

forecast_size

126

### ARIMAX с dummy регрессорами

🐎 **Какие порядки (p,d,q) подобраны для ARIMA без регрессоров?**

In [25]:
# Обучаем модель Auto ARIMA
model = auto_arima(train, stepwise=True, trace=True)

# Выводим сводку модели
print(model.summary())

# Делаем прогноз на следующие периоды
forecast_auto_arima = model.predict(n_periods=forecast_size)


forecast_auto_arima = pd.DataFrame(forecast_auto_arima)
forecast_auto_arima['ds'] = test.index
forecast_auto_arima.set_index('ds', inplace=True)
forecast_auto_arima.columns = ['y_hat']

display(forecast_auto_arima)

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=6950.070, Time=2.38 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=6951.030, Time=0.05 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=6952.548, Time=0.17 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=6952.495, Time=0.29 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=6950.995, Time=0.06 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=6953.204, Time=1.46 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=6953.046, Time=2.04 sec
 ARIMA(3,1,2)(0,0,0)[0] intercept   : AIC=6954.240, Time=1.71 sec
 ARIMA(2,1,3)(0,0,0)[0] intercept   : AIC=6954.107, Time=2.58 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=6952.589, Time=0.63 sec
 ARIMA(1,1,3)(0,0,0)[0] intercept   : AIC=6954.336, Time=1.63 sec
 ARIMA(3,1,1)(0,0,0)[0] intercept   : AIC=6954.094, Time=0.89 sec
 ARIMA(3,1,3)(0,0,0)[0] intercept   : AIC=inf, Time=3.91 sec
 ARIMA(2,1,2)(0,0,0)[0]             : AIC=6950.158, Time=2.19 sec

Best model:  ARIMA(2,1,2)(0,0,0)[0] i

,y_hat
ds,
2025-05-20,458.673264
2025-05-21,459.935069
2025-05-22,461.229869
2025-05-23,461.120750
2025-05-27,460.084306
...,...
2025-11-11,486.414241
2025-11-12,486.776945
2025-11-13,487.312482


🐎 **Какие порядки (p,d,q) подобраны для ARIMAX (с регрессорами)?**

In [26]:
# Обучаем модель Auto ARIMA
model = auto_arima(train, X=regressors[['rising_interest_rates', 'ai_boom', 'new_ai_boom']][:train_size], stepwise=True, trace=True)

# Выводим сводку модели
print(model.summary())

# Прогнозируем следующие периоды
forecast_reg_1 = model.predict(n_periods=forecast_size, X=regressors[['rising_interest_rates', 'ai_boom', 'new_ai_boom']][train_size:])

forecast_reg_1 = pd.DataFrame(forecast_reg_1)
forecast_reg_1['ds'] = test.index
forecast_reg_1.set_index('ds', inplace=True)
forecast_reg_1.columns = ['y_hat']
display(forecast_reg_1)

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(0,0,0)[0] intercept   : AIC=6955.763, Time=3.26 sec
 ARIMA(0,1,0)(0,0,0)[0] intercept   : AIC=6956.528, Time=0.12 sec
 ARIMA(1,1,0)(0,0,0)[0] intercept   : AIC=6958.005, Time=0.41 sec
 ARIMA(0,1,1)(0,0,0)[0] intercept   : AIC=6957.939, Time=2.12 sec
 ARIMA(0,1,0)(0,0,0)[0]             : AIC=6956.544, Time=0.57 sec
 ARIMA(1,1,2)(0,0,0)[0] intercept   : AIC=6958.633, Time=0.82 sec
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=6958.477, Time=1.54 sec
 ARIMA(3,1,2)(0,0,0)[0] intercept   : AIC=6959.675, Time=2.80 sec
 ARIMA(2,1,3)(0,0,0)[0] intercept   : AIC=6959.532, Time=3.04 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=6957.950, Time=1.61 sec
 ARIMA(1,1,3)(0,0,0)[0] intercept   : AIC=6959.722, Time=2.72 sec
 ARIMA(3,1,1)(0,0,0)[0] intercept   : AIC=6959.464, Time=2.48 sec
 ARIMA(3,1,3)(0,0,0)[0] intercept   : AIC=6959.578, Time=4.95 sec
 ARIMA(2,1,2)(0,0,0)[0]             : AIC=6955.717, Time=3.06 sec
 ARIMA(1,1,2)(0,0,0)[0]          

,y_hat
ds,
2025-05-20,458.427522
2025-05-21,459.452112
2025-05-22,460.538401
2025-05-23,460.229801
2025-05-27,458.970916
...,...
2025-11-11,459.234134
2025-11-12,459.369912
2025-11-13,459.697386


Посмотрим на прогнозы и проанализируем их.

In [27]:
fig = px.line(title="Microsoft closing prices (daily)")
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=forecast_auto_arima.index, y=forecast_auto_arima['y_hat'], mode='lines', name='forecast_no_regs', line=dict(color='red'))
fig.add_scatter(x=forecast_reg_1.index, y=forecast_reg_1['y_hat'], mode='lines', name='forecast_with_regs', line=dict(color='orange'))


fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

🐎 **Что можете сказать о влиянии dummy регрессоров на прогноз?**

### Анализ остатков

На занятиях 5–6 мы обсуждали, почему важно анализировать остатки модели.

🐎 **Что такое остатки модели? Как они должны вести себя в идеале?**

Метод `model.plot_diagnostics()` строит набор диагностических графиков для анализа остатков и оценки того, насколько хорошо модель ARIMA соответствует данным. Эти графики помогают выявить проблемы с остатками, качеством подгонки и предпосылками модели. Ниже приведены пояснения для каждого графика.

In [ ]:
fig = model.plot_diagnostics(figsize=(10,8))

Рассмотрим этот набор графиков подробнее.

**График стандартизированных остатков** — левый верхний.

Показывает остатки (ошибки) модели во времени, поделённые на стандартное отклонение. Это упрощает интерпретацию и сравнение.

Для $e_t = y_t - \hat{y_t}$:

$Standardized\_e_t = \frac{e_t}{σ^2}$

*Зачем стандартизировать остатки?*

* Нестандартизированные остатки зависят от масштаба данных. Например, ошибки прогноза продаж в миллионах будут намного больше, чем ошибки прогноза температуры. Стандартизация делает остатки безразмерными, что облегчает сравнение разных наборов данных.
* Стандартизированные остатки помогают быстро выявлять аномалии: значение за пределами ±3 — повод разобраться, где модель ошибается или где в данных произошли необычные события.

*На что смотреть?*

* Остатки должны выглядеть как случайный шум: распределяться вокруг нуля без явных трендов или структур.
* Сильные отклонения от нуля говорят, что модель не ловит какие-то паттерны.

*В нашем случае*

Остатки выглядят нормально.

**Гистограмма с оценкой плотности** — правый верхний график.

Показывает гистограмму остатков с наложенной ядерной оценкой плотности (KDE) и кривой нормального распределения.

* Гистограмма:
  * Отражает, как часто встречаются остатки в разных интервалах.
  * Помогает понять форму распределения (нормальная, скошенная, многомодальная и т.д.).

* KDE (ядерная оценка плотности):
  * Плавная кривая, описывающая оценочную плотность распределения остатков.
  * Даёт непрерывное приближение гистограммы и выявляет паттерны, которые не всегда видны по столбикам.

*Зачем проверять нормальность?*

Многие модели (например, ARIMA) предполагают, что остатки распределены нормально.

*На что смотреть?*

* Остатки должны напоминать колокол нормального распределения.
* Сильная асимметрия или тяжёлые хвосты могут означать, что модель объясняет данные не полностью, либо целевую переменную стоит трансформировать.

*В нашем случае*

Возможно, модель переобучается на отдельных участках: где-то остатки очень маленькие, а где-то наоборот — большие.

**Normal Q-Q Plot** — левый нижний.

График помогает проверить, следуют ли остатки нормальному распределению. Он сравнивает квантили выборочных данных с квантилями теоретического нормального распределения.

*Квантиль* — значение, делящее данные на интервалы с равной вероятностью. Например, 50‑й квантиль — это медиана, 25‑й и 75‑й квантили — первый и третий квартили. На графике квантиль остатков сопоставляется с соответствующим квантилем стандартного нормального распределения (среднее 0, дисперсия 1).

*На что смотреть?*

* Если остатки нормальны, точки лежат примерно на прямой (обычно под углом 45°).
* Отклонения от прямой указывают на ненормальность (асимметрию, выбросы и т.д.).

*В нашем случае*

Изгибы на концах графика (верхний и нижний хвосты) намекают на тяжёлые хвосты: данные могут содержать экстремальные значения чаще, чем предполагает нормальное распределение.



**Коррелограмма** — правый нижний график.

Показывает автокорреляционную функцию (ACF) остатков, то есть зависимость остатков от самих себя на разных лагах.

ACF мы уже разбирали ранее.

### ARIMAX с непрерывными регрессорами

Теперь посмотрим, как ARIMA работает с непрерывными регрессорами.

In [28]:
# Обучаем модель Auto ARIMA
model = auto_arima(train, X=stocks_new[['x']][:train_size], stepwise=True, trace=True)

# Выводим сводку модели
print(model.summary())

# Строим прогноз на следующие периоды
forecast_reg_cont = model.predict(n_periods=forecast_size, X=stocks_new[['x']][train_size:])

print(forecast_reg_cont)

Performing stepwise search to minimize aic
 ARIMA(2,0,2)(0,0,0)[0] intercept   : AIC=6521.915, Time=0.99 sec
 ARIMA(0,0,0)(0,0,0)[0] intercept   : AIC=6517.957, Time=0.43 sec
 ARIMA(1,0,0)(0,0,0)[0] intercept   : AIC=6516.971, Time=0.78 sec
 ARIMA(0,0,1)(0,0,0)[0] intercept   : AIC=6518.201, Time=0.34 sec
 ARIMA(0,0,0)(0,0,0)[0]             : AIC=6517.504, Time=0.69 sec
 ARIMA(2,0,0)(0,0,0)[0] intercept   : AIC=6520.174, Time=0.45 sec
 ARIMA(1,0,1)(0,0,0)[0] intercept   : AIC=6519.191, Time=0.53 sec
 ARIMA(2,0,1)(0,0,0)[0] intercept   : AIC=6520.753, Time=2.45 sec
 ARIMA(1,0,0)(0,0,0)[0]             : AIC=6516.433, Time=0.19 sec
 ARIMA(2,0,0)(0,0,0)[0]             : AIC=6518.322, Time=0.14 sec
 ARIMA(1,0,1)(0,0,0)[0]             : AIC=6517.431, Time=0.77 sec
 ARIMA(0,0,1)(0,0,0)[0]             : AIC=6516.364, Time=0.12 sec
 ARIMA(0,0,2)(0,0,0)[0]             : AIC=6518.174, Time=0.45 sec
 ARIMA(1,0,2)(0,0,0)[0]             : AIC=6520.314, Time=0.86 sec

Best model:  ARIMA(0,0,1)(0,0,0)

In [29]:
forecast_reg_cont = pd.DataFrame(forecast_reg_cont)
forecast_reg_cont['ds'] = test.index
forecast_reg_cont.set_index('ds', inplace=True)
forecast_reg_cont.columns = ['y_hat']

forecast_reg_cont

,y_hat
ds,
2025-05-20,455.293109
2025-05-21,454.697451
2025-05-22,455.077558
2025-05-23,450.106164
2025-05-27,456.607987
...,...
2025-11-11,504.941535
2025-11-12,509.497812
2025-11-13,510.453079


In [30]:
fig = px.line(title="Microsoft closing prices (daily)")
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=forecast_auto_arima.index, y=forecast_auto_arima['y_hat'], mode='lines', name='forecast_no_regs', line=dict(color='red'))
fig.add_scatter(x=forecast_reg_1.index, y=forecast_reg_1['y_hat'], mode='lines', name='forecast_with_dummy_regs', line=dict(color='orange'))
fig.add_scatter(x=forecast_reg_cont.index, y=forecast_reg_cont['y_hat'], mode='lines', name='forecast_with_continuous_regs', line=dict(color='hotpink'))


fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

🐎 **Как непрерывные регрессоры повлияли на прогноз?**

🐎 **Какие непрерывные регрессоры могут быть полезны для прогнозирования цен акций?**

🐎 Задание на 20 минут

**Прогнозируйте акции с помощью Auto-ARIMAX с регрессорами.**

**1. Постройте Auto-ARIMAX с dummy регрессорами для торговой войны, COVID-19 и пост-COVID периода.**

Не используйте цены открытия!!!

**2. Сделайте прогноз вне выборки с Auto-ARIMAX и непрерывным регрессором — квартальной чистой прибылью Microsoft.** Подумайте, как согласовать частоту регрессора и временного ряда.

Данные по чистой прибыли найдёте в [microsoft_net_income.csv](https://drive.google.com/file/d/1CsJV4mkWgIliQY58DUhhtNoId6yVK5KN/view?usp=sharing).


In [ ]:
net_income = pd.read_csv('microsoft_net_income.csv', sep=';', parse_dates=['quarter'])
net_income.rename(columns={'quarter':'ds', 'net_income (Millions of US $)':'x'}, inplace=True)
net_income['x'] = net_income['x'].apply(lambda x: x[1:].replace(',','')).astype(float)
net_income = net_income[['ds','x']].sort_values(by='ds').set_index('ds')
net_income

## Как выбирать регрессоры?

**Какие проблемы возникают, если добавить нерелевантные или слишком многочисленные регрессоры в модели семейства ARIMA?**

Лишние регрессоры ухудшают качество и надёжность модели. Основные причины:

1. **Переобучение.** Чем больше ненужных факторов, тем сложнее модель и тем выше шанс подогнать шум вместо закономерности.
2. **Рост дисперсии прогноза.** Модель пытается объяснить шум, коэффициенты становятся нестабильными, прогнозы остро реагируют на малейшие изменения входных данных.
3. **Мультиколлинеарность.** Сильно коррелирующие регрессоры затрудняют интерпретацию и увеличивают неопределённость оценок.
4. **Потеря интерпретируемости.** ARIMAX ценна тем, что видно влияние экзогенных факторов. Ненужные признаки усложняют трактовку и могут привести к ложным выводам.
5. **Пустая трата ресурсов.** Дополнительные регрессоры увеличивают время обучения и вычислительные затраты без гарантированного выигрыша в точности.
6. **Потенциальное смещение.** Коррелирующие нерелевантные признаки ухудшают баланс смещения/дисперсии: модель становится сложной и хуже обобщает.

Выбор правильных регрессоров критичен для точности и интерпретируемости модели.

**Как выбирать регрессоры?**

* **Здравый смысл**:
  
  * Выбирайте переменные с логичной связью с целевым рядом.
  * Опирайтесь на экспертные знания и исследования.
  * Продумайте причинно-следственный механизм влияния.

* **Графики**:
  * Постройте целевой ряд и регрессор на одном графике и сравните их.

Мы уже решили, что чистая прибыль — логичный регрессор для цен акций.
Давайте построим график и сравним её с ценами акций.

In [32]:
net_income

NameError: name 'net_income' is not defined

Регрессор «чистая прибыль» имеет квартальную частоту, а наш целевой ряд — дневной.

🐎 **Как преобразовать регрессор, чтобы использовать его в ARIMAX?**

In [31]:
date_range = pd.date_range(start=stocks.index.min(), end=stocks.index.max())

# Создаём DataFrame
net_income_daily = pd.DataFrame(date_range, columns=["ds"])

net_income_daily = net_income_daily.set_index('ds').join(net_income, how='outer').fillna(method='ffill').loc[stocks.index.min():stocks.index.max()]

net_income_daily

NameError: name 'net_income' is not defined

In [ ]:
fig = px.line(title="Microsoft closing prices (daily)")
fig.add_scatter(x=stocks.index, y=stocks['y'], mode='lines', name='y', line=dict(color='blue'))
fig.add_scatter(x=net_income_daily.index, y=net_income_daily['x'], mode='lines', name='net income', line=dict(color='red'))


fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

В этом (и во многих похожих) случае удобно использовать **графики с двумя осями Y**. Они нужны, когда два ряда имеют разный масштаб, но общую ось X (обычно время). Одна ось Y размещается слева, другая — справа, что позволяет одновременно сравнивать две величины и сохранять читаемость.

Построим такой график.

Полученный рисунок помогает лучше понять связь между целевой переменной и регрессором.

In [ ]:
import plotly.graph_objects as go

# Создаём пустой график
fig = go.Figure()

# Добавляем ряд y на левую ось
fig.add_trace(go.Scatter(
    x=stocks.index,
    y=stocks['y'],
    mode='lines',
    name='y',
    line=dict(color='blue'),
    yaxis='y1'  # Привязываем к оси y1
))

# Добавляем ряд x на правую ось
fig.add_trace(go.Scatter(
    x=net_income_daily.index,
    y=net_income_daily['x'],
    mode='lines',
    name='net income',
    line=dict(color='red'),
    yaxis='y2'  # Привязываем к оси y2
))


fig.update_layout(
    title="Microsoft closing prices (daily)",
    template='plotly_white',
    width=1000,
    height=500,
    yaxis=dict(
        title='stock prices',
    ),
    yaxis2=dict(
        title='net income',
        overlaying='y',  # Накладываем оси друг на друга
        side='right'
    ),
    xaxis=dict(title='Date')
)

# Отображаем график
fig.show()

🐎 **Считаете ли вы, что чистая прибыль — полезный непрерывный регрессор для цен акций?**

Да. Чистая прибыль хорошо отражает будущий рост или падение цен акций.

**Как отбирать регрессоры?**

* **«Пакетный» подход**:
  * обучаем ARIMA со всем набором экзогенных переменных и смотрим t-статистики для каждой;
  * удаляем регрессоры с t-статистикой ниже заранее заданного порога;
  * снова обучаем модель с оставшимися регрессорами и повторно проверяем t-статистики, убирая незначимые признаки;
  * повторяем процедуру, пока в модели не останутся только значимые регрессоры.

Теперь посмотрим на значимость коэффициентов в модели ARIMAX.

In [ ]:
print(model.summary())

🐎 **Считаете ли вы чистую прибыль полезным непрерывным регрессором для цен акций?**

**Как отбирать регрессоры?**


* **Корреляции**. О корреляциях говорят часто, но чтобы глубоко разобраться, нужно изучить векторные модели. Полезные ключевые слова: «cross-correlation», «causality», «vector autoregressions».

## Итоги по регрессорам

- Обсудили dummy регрессоры (1 — событие, 0 — остальное) и непрерывные регрессоры (например, чистая прибыль, цена на нефть)
- Применили модели ARIMAX для прогнозов с регрессорами
- Разобрали подходы к выбору подходящих регрессоров

# SARIMAX

Прогнозирование сезонных временных рядов критично, потому что продажи, спрос, погода и многие другие процессы подчиняются предсказуемым циклам. Точные прогнозы помогают компаниям оптимизировать ресурсы, улучшать планирование, снижать издержки и лучше удовлетворять клиентов. Эффективное прогнозирование позволяет действовать проактивно, адаптироваться к ожидаемым колебаниям и извлекать выгоду из повторяющихся трендов.

## Данные

Далее мы будем прогнозировать продажи (unit sales) тысяч товаров, продающихся в магазинах сети «Favorita» в Эквадоре. Обучающий датасет содержит даты, информацию о магазине и товаре, признаки промо-акций, а также объёмы продаж. Дополнительные файлы дают вспомогательные признаки, полезные при построении моделей.

Данные взяты с [Kaggle](https://www.kaggle.com/competitions/favorita-grocery-sales-forecasting/data?select=train.csv.7z). Скачать набор можно [здесь](https://drive.google.com/file/d/1xF5HXH0_fJlSr82A0nRehu6NVeq6mvhR/view?usp=sharing).

In [ ]:
# # Подготовка данных с Kaggle — этот код запускать не нужно

# df_train = pd.read_csv('train.csv', parse_dates=['date'])
# df_stores = pd.read_csv('stores.csv')
# df_item = pd.read_csv('items.csv')

# df_train = pd.merge(df_train, df_item, on='item_nbr', how='left')
# df_train = pd.merge(df_train, df_stores, on='store_nbr', how='left')

# df_full = pd.pivot_table(df_train, values='unit_sales', index=['date','city', 'state','type','family'],
#                aggfunc='sum').reset_index()

# df_full.to_csv('train_data.csv')

In [ ]:
sales_raw = pd.read_csv('train_data.csv', parse_dates=['date'])
sales_raw

Рассмотрим суммарные продажи компании. Подготовим датасет.

В этом уроке возьмём только 2015–2017 годы, хотя в исходном наборе есть данные за 2013–2017.

In [ ]:
sales = pd.pivot_table(sales_raw, index='date', values='sales', aggfunc='sum').reset_index()
sales.columns = ['ds', 'y']
sales = sales.set_index('ds')
sales = sales.loc['2015-01-01':]
sales

Построим график данных.

In [ ]:
fig = px.line(sales, x=sales.index, y=sales['y'],
              title="Favorita sales")

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="sales")
fig.show()

Теперь разобьём данные на train/test: 2017 год пойдёт в тест, 2015–2016 — в обучение.

In [ ]:
train_size = 365*2-1
train, test = sales[:train_size], sales[train_size:]

In [ ]:
fig = px.line(title="Sales")
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test', line=dict(color='green'))

fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

🐎 **Что можно сказать об этом временном ряде продаж?**

🐎 **Что делать с рядами, в которых есть тренд и сезонность?**

## Декомпозиция

Предположим несколько сезонностей: недельную, месячную и годовую.

🐎 **Какие длины сезонных периодов выбрать?**

In [ ]:
mstl = MSTL(sales['y']) #,  periods=(?))
res = mstl.fit()

fig = res.plot()
fig.set_size_inches(10, 6)
plt.show()

res.resid.plot()

🐎 **Как понять, успешна ли декомпозиция?**

🐎 **Что можно сказать о декомпозиции, глядя на остатки?**

К сожалению, в реальных задачах мы часто получаем автокоррелированные остатки.

* **Ограничения моделей тренда и сезонности**. STL и классическая декомпозиция пытаются выделить тренд и сезонность, но далеко не всегда улавливают всю сложность временного ряда. Некоторые сезонные или циклические паттерны могут быть слишком слабыми или нерегулярными, поэтому часть структуры остаётся в остатках.

* **Взаимодействие тренда и сезонности**. Бывает, что тренд и сезонность влияют друг на друга так, что простые аддитивные/мультипликативные модели этого не видят. Например, если сезонность меняется со временем (амплитуда или форма периодов), декомпозиция может недоучесть эти изменения, и остатки сохранят автокорреляцию.

* **Пропущенные переменные**. Если на ряд действуют внешние факторы (экономика, погода, маркетинг), но мы их не включили в декомпозицию или последующие модели, остатки будут содержать эту недостающую информацию и выглядеть автокоррелированными.

## Прогноз тренда

Модели вроде auto-ARIMA помогают прогнозировать тренд после декомпозиции (в прошлых уроках мы уже это делали).

Давайте спрогнозируем трендовую компоненту.

In [ ]:
trend = res.trend.reset_index()
trend.columns = ['ds','y']
trend = trend.set_index('ds')
trend

In [ ]:
trend_train, trend_test = trend[:train_size], trend[train_size:]

In [ ]:
fig = px.line(title="Sales")
fig.add_scatter(x=trend_train.index, y=trend_train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=trend_test.index, y=trend_test['y'], mode='lines', name='test', line=dict(color='green'))

fig.update_layout(template='plotly_white', width=800, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

In [ ]:
forecast_size = len(sales) - train_size
forecast_size

In [ ]:
# Обучаем модель Auto ARIMA
model = auto_arima(trend_train, stepwise=True, trace=True)

# Выводим сводку модели
print(model.summary())

# Строим прогноз на следующие периоды
trend_forecast = model.predict(n_periods=forecast_size)


In [ ]:
trend_forecast = pd.DataFrame(trend_forecast)
trend_forecast['ds'] = trend_test.index
trend_forecast.set_index('ds', inplace=True)
trend_forecast.columns = ['y_hat']

trend_forecast

Посмотрим на получившийся прогноз тренда.

In [ ]:
fig = px.line(title="Sales")
fig.add_scatter(x=trend_train.index, y=trend_train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=trend_test.index, y=trend_test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=trend_forecast.index, y=trend_forecast['y_hat'], mode='lines', name='forecast', line=dict(color='red'))


fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

## Прогноз сезонности

Сезонность можно прогнозировать разными способами. Сегодня сосредоточимся на моделях SARIMAX.

**SARIMA** (Seasonal Autoregressive Integrated Moving Average) — расширение ARIMA, в которое добавлены сезонные компоненты, чтобы лучше описывать периодические паттерны. Разберёмся по шагам.

Прежде чем перейти к SARIMA, вспомним, из чего состоит **ARIMA**:

* **AR (авторегрессия, p)**: использует прошлые значения ряда (лаги) для прогнозирования будущего.
* **I (интегрирование, d)**: применяет дифференцирование, чтобы сделать ряд стационарным и убрать тренд.
* **MA (скользящее среднее, q)**: учитывает прошлые ошибки прогноза для улучшения текущего предсказания.

SARIMA повторяет идею ARIMA, но добавляет сезонность:

* **Сезонные компоненты (P, D, Q)**: аналогичны несезонным, но описывают сезонную часть ряда.

Формально:

$$SARIMA(p,d,q)\times(P,D,Q)_m$$

* **p, d, q** — привычные параметры ARIMA(p,d,q).
* **P, D, Q** — порядок сезонной авторегрессии, сезонного дифференцирования и сезонного скользящего среднего.
* **m** — длина сезонного периода (например, 12 для месячных данных с годовым циклом).

**Seasonal AR (P)** — связь текущего наблюдения с значением через сезонный лаг (например, значение 12 месяцев назад).

**Seasonal Differencing (D)** — сезонное дифференцирование, то есть вычитание значения, наблюдавшегося в том же периоде прошлого сезона. Так мы убираем сезонность и облегчает моделирование.

**Seasonal MA (Q)** — аналог сезонного скользящего среднего: использует ошибки предыдущих сезонов для улучшения прогноза.

*Пример.* Месячные продажи с годовой сезонностью (m = 12).

* **m = 12**: месячные данные с годовым циклом (январь–декабрь).
* **P = 1**: используем продажи из того же месяца прошлого года.
* **D = 1**: делаем сезонное дифференцирование, чтобы стационизировать ряд: $Y'_t = Y_t - Y_{t-s}$.
* **Q = 1**: корректируем прогноз ошибками предыдущего сезона.

In [ ]:
# Два графика рядом
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# ACF (автокорреляция)
plot_acf(train['y'].dropna(), ax=ax[0], lags=20)
ax[0].set_title('ACF')

# PACF (частичная автокорреляция)
plot_pacf(train['y'].dropna(), ax=ax[1], lags=20)
ax[1].set_title('PACF')


plt.tight_layout()
plt.show()

In [ ]:
sarima_model = auto_arima(train['y'],
                          seasonal=True,
                          m=7,  # недельная сезонность
                          max_p=3,
                          max_q=3,
                          trace=True,
                          error_action='ignore',
                          suppress_warnings=True,
                          stepwise=True)
print(sarima_model.summary())

In [ ]:
forecast = sarima_model.predict(n_periods=forecast_size)

;;

In [ ]:
forecast = pd.DataFrame(forecast)
forecast['ds'] = test.index
forecast.set_index('ds', inplace=True)
forecast.columns = ['y_hat']

forecast

In [ ]:
fig = px.line(title="Sales")
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train', line=dict(color='blue'))
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test', line=dict(color='green'))
fig.add_scatter(x=forecast.index, y=forecast['y_hat'], mode='lines', name='forecast', line=dict(color='red'))


fig.update_layout(template='plotly_white', width=1000, height=500)
fig.update_xaxes(title_text="date")
fig.update_yaxes(title_text="closing price")
fig.show()

🐎 Задание на 20 минут

**Спрогнозируйте продажи Favorita с помощью регрессоров.**

**1. Подготовьте dummy регрессоры для праздников.**

**2. Подготовьте непрерывные регрессоры по ценам на нефть.**

**3. Постройте прогноз с SARIMAX и регрессорами (цены на нефть + праздники).**
Данные по нефти возьмите из [oil.csv](https://drive.google.com/file/d/1D4rnInuBSKsYkaQl5yi9DKpUuKtrwIAQ/view?usp=drive_link), по праздникам — из [holidays_events.csv](https://drive.google.com/file/d/1cWdWO8CevKUPU-vqU58MhGun6_Z8_9kA/view?usp=drive_link).


In [ ]:
oil = pd.read_csv('oil.csv', parse_dates=['date'])
oil.head()

In [ ]:
holidays = pd.read_csv('holidays_events.csv', parse_dates=['date'])
holidays

## Итоги по SARIMAX

- Провели декомпозицию MSTL и выделили тренд и сезонности
- Прогнозировали тренд с помощью auto-ARIMAX
- Обсудили модель SARIMAX для рядов с сезонностью

# Метрики качества

**Метрики качества** в прогнозировании временных рядов позволяют количественно оценивать точность и эффективность модели. С их помощью мы сравниваем модели, выбираем подходящую для задачи и находим слабые места процесса прогнозирования.

## Масштаб-зависимые ошибки

$$\text{Средняя абсолютная ошибка (MAE)} = mean(|y_t - \hat{y_t}|)$$
$$\text{Среднеквадратичная ошибка (RMSE)} = \sqrt{mean((y_t - \hat{y_t})^2)}$$

Важно: **RMSE** минимизирует среднее квадратичное отклонение между фактом и прогнозом, а **MAE** больше ориентирована на медианное отклонение.

Отсюда следует, что **RMSE сильнее наказывает** за крупные ошибки, чем MAE.



In [ ]:
# Фактические значения
y_actual = np.array([10, 15, 20, 25, 30])

# Прогноз: сценарий 1 — небольшие систематические ошибки
y_pred_small_errors = np.array([11, 14, 21, 24, 29])

# Прогноз: сценарий 2 — один большой выброс
y_pred_outlier = np.array([11, 14, 21, 24, 50])  # выброс в последнем значении

# Считаем MAE и RMSE для небольших ошибок
mae_small = np.mean(np.abs(y_actual - y_pred_small_errors))
rmse_small = np.sqrt(np.mean((y_actual - y_pred_small_errors) ** 2))

# Считаем MAE и RMSE для сценария с выбросом
mae_outlier = np.mean(np.abs(y_actual - y_pred_outlier))
rmse_outlier = np.sqrt(np.mean((y_actual - y_pred_outlier) ** 2))

# Выводим результаты
results = pd.DataFrame({
    "Scenario": ["Small Errors", "Outlier"],
    "MAE": [mae_small, mae_outlier],
    "RMSE": [rmse_small, rmse_outlier]
})


plt.figure(figsize=(8, 6))

# Фактический ряд
plt.plot(y_actual, label="Actual Values", color="black", linewidth=2)

# Прогноз со сплошными мелкими ошибками
plt.plot(y_pred_small_errors, label="Predicted (Small Errors)", linestyle="--", color="blue", linewidth=2)

# Прогноз с выбросом
plt.plot(y_pred_outlier, label="Predicted (With Outlier)", linestyle="--", color="red", linewidth=2)


print(results)

Посмотрим на ошибки прогноза без регрессоров.

In [ ]:
print('MAE: ', np.mean(np.abs(test['y']-forecast['y_hat'])))
print('RMSE: ', np.sqrt(np.mean((test['y']-forecast['y_hat'])**2)))

## Масштаб-независимые ошибки

$$\text{Средняя абсолютная процентная ошибка (MAPE)} = mean\left(\frac{|y_t - \hat{y_t}|}{y_t}\right)$$

Несмотря на простую интерпретацию, у **MAPE** есть два серьёзных недостатка:

* Если $y_t$ близок к нулю, ошибка резко возрастает или вовсе становится нечислимой.

* Когда прогноз завышает фактическое значение (Forecast > Actual), знаменатель MAPE равен факту, поэтому ошибка растёт неограниченно: нижний предел — 100%, верхний — бесконечность. Метрика асимметрична.



In [ ]:
# Фактические значения (y_actual)
y_actual = np.array([10, 20, 30])

# Сценарий 1: сильное завышение прогноза
y_pred_over = np.array([10+100, 20+200, 30+300])

# Сценарий 2: умеренное занижение
y_pred_under = np.array([10-5, 20-10, 30-15])

# Сценарий 3: сильное занижение (уход в отрицательные значения)
y_pred_under_zero = np.array([10-20, 20-30, 30-40])

# Считаем MAPE для всех сценариев
mape_over = np.mean(np.abs((y_actual - y_pred_over) / y_actual)) * 100
mape_under = np.mean(np.abs((y_actual - y_pred_under) / y_actual)) * 100
mape_under_zero = np.mean(np.abs((y_actual - y_pred_under_zero) / y_actual)) * 100

# Выводим результаты
results = pd.DataFrame({
    "Scenario": ["Overestimation", "Underestimation", "Underestimation (negative)"],
    "MAPE (%)": [mape_over, mape_under, mape_under_zero]
})
print(results)

# Визуализируем
plt.figure(figsize=(10, 6))

# Фактический ряд
plt.plot(y_actual, label="Actual Values", color="black", linewidth=2)

# Прогноз с завышением
plt.plot(y_pred_over, label="Forecast (Overestimate)", linestyle="--", color="red", linewidth=2)

# Прогноз с занижением
plt.plot(y_pred_under, label="Forecast (Underestimate)", linestyle="--", color="blue", linewidth=2)

# Прогноз с сильным занижением (отрицательные значения)
plt.plot(y_pred_under_zero, label="Forecast (Underestimate, negative)", linestyle="--", color="green", linewidth=2)

Разберём ошибки прогноза без регрессоров.

In [ ]:
print('MAPE: ', np.mean(np.abs((test['y'] - forecast['y_hat']) / test['y'])) * 100)

## Прочие ошибки

**MDAPE** (Median Absolute Percentage Error) — модификация MAPE, которая даёт более устойчивую оценку при наличии выбросов или больших отклонений. Главное отличие — вместо среднего берётся медиана абсолютных процентных ошибок.

$$\text{Медианная абсолютная процентная ошибка: }  MDAPE = median\left(\frac{|y_t - \hat{y_t}|}{y_t}\right)$$

In [ ]:

# Пример данных
y_actual = np.array([100, 200, 300, 400, 500])  # фактические значения
y_pred = np.array([110, 210, 280, 350, 5000])    # прогнозируемые значения

# Рассчитаем MAPE
mape = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100
print("MAPE: ", mape)

# Рассчитаем MDAPE
mdape = np.median(np.abs((y_actual - y_pred) / y_actual)) * 100
print("MDAPE: ", mdape)

print("Abs diffs: " , np.abs(y_actual-y_pred))

Посмотрим на ошибки прогноза без регрессоров.

In [ ]:
print('MDAPE: ', np.median(np.abs((test['y'] - forecast['y_hat']) / test['y'])) * 100)

Используйте MDAPE, когда:

* в данных есть выбросы или большие отклонения;
* нужна стабильная оценка точности, нечувствительная к редким, но крупным ошибкам.

**MASE** (Mean Absolute Scaled Error) — ещё одна метрика ошибок в прогнозировании временных рядов. Она устраняет недостатки классических метрик вроде MAPE или RMSE, особенно если нужно сравнивать точность на разных рядах или шкалах.

$$\text{Средняя абсолютная масштабированная ошибка (для сезонных данных): }  MASE = mean\left(\frac{|y_t - \hat{y_t}|}{\frac{1}{T-m}\sum^T_{t=m+1}|y_t-y_{t-m}|}\right)$$

Знаменатель — средняя абсолютная ошибка наивного прогноза. Наивный прогноз предполагает, что значение в момент $t$ равно значению в момент $t-1$. Это и есть отправная точка для сравнения.

Разберём ошибки прогноза без регрессоров.

In [ ]:
print('MASE: ', np.mean(np.abs(test['y']-forecast['y_hat']))/np.mean(np.abs(test['y']-test['y'].shift(1))))

MASE > 1 означает, что ошибка выбранного прогноза больше, чем ошибка наивного прогноза, т.е. выбранный прогноз хуже, чем наивный.

MASE < 1, с другой стороны, указывает на обратное.

**SMAPE** (Симметричная средняя абсолютная процентная ошибка) - это показатель, используемый для оценки точности прогнозных моделей. Он аналогичен MAPE, но устраняет некоторые ограничения MAPE, такие как проблемы, когда фактические значения близки к нулю.

 $$\text{ Симметричная средняя абсолютная процентная ошибка: }  SMAPE = mean(\frac{2*|y_t - \hat{y_t}|}{|y_t|+ |\hat{y_t}|})$$

Проанализируем ошибки прогноза без регрессоров.

In [ ]:
print('SMAPE: ', np.mean(2*np.abs(test['y']-forecast['y_hat'])/(test['y']+forecast['y_hat'])))

## Итоги по метрикам

| **Метрика** | **Лучший сценарий** | **Преимущества** | **Недостатки** | **Когда применять** |
|------------|-------------------|----------------|-------------------|-----------------|
| **MAE** | Простая и понятная оценка величины ошибки. | - Легко трактовать.<br>- Слабо реагирует на выбросы. | - Слабо штрафует крупные ошибки (по сравнению с RMSE). | - Когда хотите одинаково относиться ко всем ошибкам. |
| **RMSE** | Нужно сильнее штрафовать большие отклонения. | - Сильнее наказывает крупные ошибки.<br>- Стандартная метрика в регрессии. | - Чувствителен к выбросам.<br>- Измеряется в квадратных единицах. | - Когда большие отклонения особенно критичны. |
| **MAPE** | Требуются ошибки в процентах и нет значений, близких к нулю. | - Интуитивна.<br>- Процентная интерпретация. | - Резко растёт возле нуля.<br>- Асимметрична: перепрогноз даёт огромные ошибки. | - Общие задачи, где важно выражать ошибки в процентах. |
| **MDAPE** | Нужно снизить влияние редких больших выбросов. | - Менее чувствительна к выбросам, чем MAPE.<br>- Более устойчива при перекошенных данных. | - Чуть сложнее трактовать (использует медиану). | - Когда данные содержат выбросы или сильно асимметричны. |
| **MASE** | Нужно сравнить модель с наивным прогнозом или между датасетами. | - Нормируется на наивный прогноз.<br>- Позволяет сравнивать ряды разного масштаба. | - Менее интуитивен, чем MAE/RMSE.<br>- Зависит от выбранного наивного прогноза. | - Когда важно понять, лучше ли модель простого baseline. |
| **SMAPE** | Нужна симметричная процентная метрика. | - Одинаково штрафует перепрогноз и недопрогноз.<br>- Лучше работает при значениях, близких к нулю. | - Может вести себя плохо, если и факт, и прогноз ≈ 0.<br>- Иногда сложна для интерпретации. | - Когда нужно одинаково относиться к завышениям и занижениям, а в данных возможны нули. |


Итого:

* **MAE** — простая метрика, одинаково относящаяся ко всем ошибкам.
* **RMSE** — усиливает вес крупных отклонений и чувствительна к выбросам.
* **MAPE** — ошибки в процентах, но плохо работает возле нуля.
* **MDAPE** — более устойчива, если есть выбросы или перекошенные распределения.
* **MASE** — удобна для сравнения с наивным прогнозом и между датасетами.
* **SMAPE** — симметричная процентная метрика, подходящая при наличии нулей.

В основном будем использовать RMSE и MAPE.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
print('MAE: ', np.mean(np.abs(test['y']-forecast['y_hat'])))
print('RMSE: ', np.sqrt(np.mean((test['y']-forecast['y_hat'])**2)))

🐎 Задание на 10 минут

**Оцените свой прогноз продаж (с регрессорами) по всем перечисленным метрикам.**

# ИТОГ

- Обсудили dummy регрессоры (1 — событие, 0 — иначе) и непрерывные регрессоры (например, чистая прибыль, цена на нефть)
- Применили модели ARIMAX для прогнозирования с регрессорами
- Разобрали подходы к выбору релевантных регрессоров
- Рассмотрели модель SARIMAX для сезонных рядов
- Обсудили масштаб-зависимые, масштаб-независимые и другие метрики качества
- Поговорили о выборе подходящей метрики качества

Пожалуйста, оставьте отзыв [здесь](https://forms.gle/bh49kMtdd1driCzt6), чтобы сделать занятия ещё эффективнее :)